In [1]:
from __future__ import annotations

import base64
import json
import mimetypes
import re
from pathlib import Path
from typing import Any

import cv2
import supervision as sv
from openai import OpenAI
from ultralytics import YOLOE

In [2]:
YOLO_MODEL = "yoloe-26x-seg.pt"

QWEN3VL_MODEL = "docker.io/ai/qwen3-vl:2B-UD-Q4_K_XL"
BASE_URL = "http://localhost:12434/engines/v1"
API_KEY = "not-needed"
MAX_TOKENS = 512
TEMPERATURE = 0.2
CONFIDENCE = 0.25

In [3]:
GARBAGE_CLASSES = [
    "trash can",
    "trash bin",
    "spray can",
    "bag",
    "toilet paper",
    "beer bottle",
    "beer can",
    "bottle",
    "bottle cap",
    "can",
    "cardboard",
    "cardboard box",
    "carton",
    "cigar",
    "cigarette",
    "coffee cup",
    "cup",
    "debris",
    "diaper",
    "paper cup",
    "garbage",
    "glass bottle",
    "glass jar",
    "grocery bag",
    "leftover",
    "napkin",
    "paper",
    "paper bag",
    "paper plate",
    "paper towel",
    "plastic",
    "rubble",
    "scrap",
    "shopping bag",
    "tin",
    "tinfoil",
    "tissue",
    "waste",
    "wine bottle",
    "wrapping paper",
]

PET_CLASSES = ["dog", "cat"]

ALL_CLASSES = PET_CLASSES + GARBAGE_CLASSES

In [4]:
GARBAGE_PROMPT = """Analyze the provided image of a public trash can and classify its status as [Safe/Healthy] or [Unsafe/Unhealthy].

Use the following criteria for your evaluation:

Containment: Is all waste inside the bin, or is there overflow and litter on the ground within a 2-meter radius?

Structural Integrity: Is the bin upright, properly secured, and free of significant physical damage?

Hygiene & Hazards: Are there signs of liquid leaks (leachate), animal scavenging, or potentially hazardous materials (sharps, chemicals) visible?

Accessibility: Is the area around the bin clear for pedestrians and maintenance workers?

Output in JSON format only:

{
  "status": "safe" | "unsafe",
  "confidence_score": 0,
  "key_observations": [
    "specific visual cue 1",
    "specific visual cue 2"
  ],
  "containment": "clear short assessment",
  "structural_integrity": "clear short assessment",
  "hygiene_hazards": "clear short assessment",
  "accessibility": "clear short assessment",
  "reason": "one short overall explanation"
}

If the bin appears normal, clean, upright, and the surrounding area is clear, return:
{
  "status": "safe",
  "confidence_score": 90,
  "key_observations": ["clear perimeter", "waste contained inside bin"],
  "containment": "Waste appears properly contained.",
  "structural_integrity": "The bin appears upright and undamaged.",
  "hygiene_hazards": "No clear hygiene or hazard issue is visible.",
  "accessibility": "The area around the bin appears clear.",
  "reason": "The trash can appears healthy and safe based on visible evidence."
}

Rules:
- status must be either "safe" or "unsafe"
- confidence_score must be an integer from 0 to 100
- key_observations must be a list of short visual observations only
- containment, structural_integrity, hygiene_hazards, and accessibility must each contain a short assessment based only on visible evidence
- reason must be one short summary sentence
- do not use any markdown formatting like ``` or add any explanation before or after the JSON
- return JSON only with no extra text"""


PET_PROMPT = """Analyze the provided image and determine whether the visible pet appears likely lost, unattended, or safely accompanied.

Use the following criteria for your evaluation:

Owner Presence: Is there a clearly visible owner, handler, leash, or direct supervision nearby?

Animal Context: Is the pet alone on a street, sidewalk, roadside, or public area?

Behavior and Position: Does the pet appear to be wandering, waiting alone, roaming near traffic, or otherwise unattended?

Safety Context: Is the pet in a potentially unsafe situation such as near vehicles, in the road, or in a confusing public environment?

Output in JSON format only:

{
  "status": "likely_lost" | "not_lost" | "uncertain",
  "confidence_score": 0,
  "key_observations": [
    "specific visual cue 1",
    "specific visual cue 2"
  ],
  "owner_presence": "clear short assessment",
  "animal_context": "clear short assessment",
  "safety_context": "clear short assessment",
  "reason": "one short overall explanation"
}

If the pet appears clearly accompanied, supervised, or not in a risky unattended context, return:
{
  "status": "not_lost",
  "confidence_score": 90,
  "key_observations": ["owner visible nearby", "pet appears supervised"],
  "owner_presence": "A likely owner or direct supervision is visible.",
  "animal_context": "The pet does not appear isolated.",
  "safety_context": "No clear unattended street risk is visible.",
  "reason": "The pet does not appear lost based on visible evidence."
}

Rules:
- status must be one of "likely_lost", "not_lost", or "uncertain"
- confidence_score must be an integer from 0 to 100
- key_observations must be a list of short visual observations only
- owner_presence, animal_context, and safety_context must each contain a short assessment based only on visible evidence
- reason must be one short summary sentence
- do not use any markdown formatting like ``` or add any explanation before or after the JSON
- return JSON only with no extra text"""

In [5]:
def image_file_to_data_url(image_path: str) -> str:
    """Read a local image file and convert it to a base64 data URL.

    Args:
        image_path: Path to a local image file.

    Returns:
        Base64 data URL for the image.

    Raises:
        FileNotFoundError: If the image file does not exist.
    """
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")

    mime_type, _ = mimetypes.guess_type(path.name)
    if mime_type is None:
        mime_type = "application/octet-stream"

    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded}"


def extract_json_from_text(text: str) -> dict[str, Any]:
    """Extract and parse the first JSON object from model output.

    Args:
        text: Raw model response text.

    Returns:
        Parsed JSON dictionary.

    Raises:
        ValueError: If no JSON object can be found.
        json.JSONDecodeError: If the extracted JSON is invalid.
    """
    cleaned = text.strip()

    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)

    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match is None:
        raise ValueError("No JSON object found in model response.")

    return json.loads(match.group(0))


def run_qwen_prompt(
    client: OpenAI,
    image_path: str,
    prompt: str,
    model_name: str,
    max_tokens: int,
    temperature: float,
) -> dict[str, Any]:
    """Run Qwen-VL on the full image with the given prompt.

    Args:
        client: OpenAI-compatible client.
        image_path: Input image path.
        prompt: Text prompt to send.
        model_name: Vision model name.
        max_tokens: Maximum output tokens.
        temperature: Sampling temperature.

    Returns:
        Parsed JSON response from the model.

    Raises:
        RuntimeError: If the model returns no content.
    """
    image_data_url = image_file_to_data_url(image_path)

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": image_data_url},
                    },
                ],
            }
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )

    if not response.choices:
        raise RuntimeError("No choices returned by the model.")

    message = response.choices[0].message
    if message is None or message.content is None:
        raise RuntimeError("Model returned an empty message.")

    return extract_json_from_text(message.content)


def get_detection_names_and_labels(results: Any, detections: sv.Detections) -> tuple[list[str], list[str]]:
    """Build class names and labels from YOLOE detections.

    Args:
        results: Raw Ultralytics result object.
        detections: Supervision detections object.

    Returns:
        Tuple of class names and display labels.
    """
    class_names: list[str] = []
    labels: list[str] = []

    if detections.class_id is None or detections.confidence is None:
        return class_names, labels

    names_map = results.names if hasattr(results, "names") else {}

    for class_id, confidence in zip(detections.class_id.tolist(), detections.confidence.tolist()):
        class_name = str(names_map.get(int(class_id), int(class_id)))
        class_names.append(class_name)
        labels.append(f"[{class_id}] {class_name} {confidence:.2f}")

    return class_names, labels


def annotate_image(
    image: Any,
    detections: sv.Detections,
    labels: list[str],
) -> Any:
    """Annotate the image with detections and labels.

    Args:
        image: Input OpenCV image.
        detections: Supervision detections.
        labels: Label strings.

    Returns:
        Annotated image.
    """
    box_annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()

    annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
    annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)
    return annotated


def split_detected_categories(class_names: list[str]) -> tuple[list[str], list[str]]:
    """Split detected class names into garbage and pet hits.

    Args:
        class_names: Detected class names.

    Returns:
        Tuple of unique garbage hits and pet hits.
    """
    garbage_hits: list[str] = []
    pet_hits: list[str] = []

    for name in class_names:
        if name in GARBAGE_CLASSES and name not in garbage_hits:
            garbage_hits.append(name)
        if name in PET_CLASSES and name not in pet_hits:
            pet_hits.append(name)

    return garbage_hits, pet_hits


def process_image(
    image_path: str,
    output_image_path: str,
    detector_model: str,
    qwen_model: str,
    base_url: str,
    api_key: str,
    conf: float,
    max_tokens: int,
    temperature: float,
) -> dict[str, Any]:
    """Run the full image pipeline: YOLOE detection then task-routed Qwen analysis.

    Args:
        image_path: Path to the input image.
        output_image_path: Path to save the annotated image.
        detector_model: Path to YOLOE weights.
        qwen_model: Qwen model name.
        base_url: OpenAI-compatible base URL.
        api_key: API key for the endpoint.
        conf: YOLOE confidence threshold.
        max_tokens: Maximum Qwen tokens.
        temperature: Qwen temperature.

    Returns:
        Final combined result dictionary.
    """
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Failed to read image: {image_path}")

    yolo_model = YOLOE(detector_model)
    yolo_model.set_classes(ALL_CLASSES)

    client = OpenAI(base_url=base_url, api_key=api_key)

    results = yolo_model.predict(source=image, conf=conf, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)

    class_names, labels = get_detection_names_and_labels(results, detections)
    annotated_image = annotate_image(image=image, detections=detections, labels=labels)

    output_path = Path(output_image_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), annotated_image)

    garbage_hits, pet_hits = split_detected_categories(class_names)

    analyses: dict[str, Any] = {}

    if garbage_hits:
        analyses["garbage"] = {
            "trigger_classes": garbage_hits,
            "qwen_result": run_qwen_prompt(
                client=client,
                image_path=image_path,
                prompt=GARBAGE_PROMPT,
                model_name=qwen_model,
                max_tokens=max_tokens,
                temperature=temperature,
            ),
        }

    if pet_hits:
        analyses["pet"] = {
            "trigger_classes": pet_hits,
            "qwen_result": run_qwen_prompt(
                client=client,
                image_path=image_path,
                prompt=PET_PROMPT,
                model_name=qwen_model,
                max_tokens=max_tokens,
                temperature=temperature,
            ),
        }

    return {
        "image_path": image_path,
        "annotated_image_path": str(output_path),
        "all_detected_classes": class_names,
        "garbage_trigger_classes": garbage_hits,
        "pet_trigger_classes": pet_hits,
        "analyses": analyses,
    }

In [ ]:
image_path = "../data/images/waste/20.jpg"
output_image = "./outputs/annotated_image.jpg"

result = process_image(
    image_path=image_path,
    output_image_path=output_image,
    detector_model=YOLO_MODEL,
    qwen_model=QWEN3VL_MODEL,
    base_url=BASE_URL,
    api_key=API_KEY,
    conf=CONFIDENCE,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "image_path": "../data/images/waste/22.jpg",
  "annotated_image_path": "outputs\\annotated_image.jpg",
  "all_detected_classes": [],
  "garbage_trigger_classes": [],
  "pet_trigger_classes": [],
  "analyses": {}
}
